# Exercise: Large-scale wind and solar generation time series 

Data for the exercise are given in four files (in subfolder “Data”):

- “Wind_generation_simulated.csv”: One year of simulated onshore wind generation data for 3 countries
- “Solar_generation_simulated.csv”: One year of simulated solar generation data for the same 3 countries
- “Wind_generation_simulated_model2.csv”: One year of simulated onshore wind generation data (only 1 country), using a different simulation model (model 2)
- “Wind_generation_measured.csv”: One year of measured onshore wind generation data (only 1 country)

All data are given in GMT time zone. All data are given as standardized generation, i.e., as share of installed capacity (0 means no generation and 1 means generation at full installed capacity).


In [ ]:
#import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance

# Define data folder
folder_data = "./Data/"

## Exercise 1: Wind generation in multiple countries

Load the simulated wind generation data for the three countries.

In [ ]:
# Load wind generation data
# Read CSV files as time-indexed DataFrames 
TT_wind_sim = pd.read_csv(folder_data + "Wind_generation_simulated.csv",
                          index_col=0, parse_dates=True)

In [ ]:
#Run this cell to see the data (you can also look directly into the csv file)
print(TT_wind_sim)

**1.1:** We want to visualize the data as time series, on hourly resolution, daily resolution (daily mean) and monthly resolution (monthly mean). The script below resamples the hourly data to daily. Complete the script to do the same for monthly.

In [ ]:
# Compute aggregate wind generation from the 3 countries (we use this later)
TT_wind_sim["Aggregate"] = TT_wind_sim.iloc[:, 0:3].mean(axis=1)

#Daily resampling where "D" stands for daily
TT_wind_sim_daily = TT_wind_sim.resample("D").mean()

#Monthly resampling
TT_wind_sim_monthly = TT_wind_sim.resample("ME").mean()

In [ ]:
#run this to see the daily resampled data
print("Daily resampled data:")
print(TT_wind_sim_daily)

In [ ]:
#run this to see the monthly resampled data
print("Monthly resampled data:")
print(TT_wind_sim_monthly)

Once the data has been resampled we can plot the different resolutions by running this cell!

In [ ]:
# Plotting 
fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=False)

# Hourly
axs[0].plot(TT_wind_sim.index, TT_wind_sim.values)
axs[0].set_ylim(0, 1)
axs[0].set_title("Onshore wind generation (hourly, GMT)")
axs[0].legend(TT_wind_sim.columns)
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Capacity Factor")

# Daily
axs[1].plot(TT_wind_sim_daily.index, TT_wind_sim_daily.values)
axs[1].set_title("Onshore wind generation (Daily)")
axs[1].set_ylim(bottom=0)
axs[1].legend(TT_wind_sim_daily.columns)
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Capacity Factor")

# Monthly
axs[2].plot(TT_wind_sim_monthly.index, TT_wind_sim_monthly.values)
axs[2].set_title("Onshore wind generation (Monthly)")
axs[2].set_ylim(bottom=0)
axs[2].legend(TT_wind_sim_monthly.columns)
axs[2].set_xlabel("Time")
axs[2].set_ylabel("Capacity Factor")

plt.tight_layout() 
plt.show()

plt.show()

**1.2:** Here we visualize the data also as PDFs (hourly data). You can use different methods (e.g., histogram based or kernel density estimation). Does this histogram approach for visualization appear appropriate for the data? Why?

Answer: *Double click to edit*

In [ ]:

fig, ax_hist = plt.subplots(1, 1, figsize=(8, 5))

for i in range(4):
    ax_hist.hist(TT_wind_sim.iloc[:, i], bins=40, density=True,
                 histtype="step")

ax_hist.set_xlim(0, 1)
ax_hist.set_title("Generation PDF (hourly)")
ax_hist.legend(TT_wind_sim.columns)
ax_hist.set_xlabel("Capacity Factor")
ax_hist.set_ylabel("Density")

plt.tight_layout()
plt.show()

**1.2:** Bonus! Here you can try to visualize using a kernel density estimation (KDE).

In [ ]:
#Fill in your code here for KDE-based visualization

**1.3:** Here we calculate mean, SD and relative SD (RSD) for each country (hourly data). Which country has the highest RSD? Can you think of any reason why? Which country has the lowest RSD?

Answer: *Double click to edit*

In [ ]:
#Run this to calculate mean, SD and RSD for each country

# Statistics
T_stats_wind = pd.DataFrame({
    "Mean": TT_wind_sim.mean(),
    "SD": TT_wind_sim.std()
})
T_stats_wind["RSD"] = T_stats_wind["SD"] / T_stats_wind["Mean"]

print("------- Stats for wind ---------")
print(T_stats_wind)

**1.4:** In our plots we also have aggregate onshore wind generation of the three countries, assuming each country has the same amount of installed capacity (aggregate generation is the sum of the generation at each time step; however, note that also the aggregate generation should be given as standardized generation). 

Consider the visualizations and descriptive statistics for the aggregate generation as for the individual countries in 1.1-1.3. How does the PDF of the aggregate generation look compared to the individual countries? Compare the RSD of the aggregate generation to the RSDs of the individual countries. How does it compare? Why?

Answer: *Double click to edit*


**1.5:** Here we calculate hourly generation ramp time series for the individual countries and for the aggregate generation. We visualize the generation ramp PDFs for the individual countries and for the aggregate and calculate the respective ramp SDs. 

How do the aggregate ramps look compared to the individual countries? Why?

Answer: *Double click to edit*

In [ ]:
#Run this to calculate and plot hourly generation ramps

# Ramp (difference)
TT_wind_sim_hourlyRamp = TT_wind_sim.diff()

# Plotting the ramp data
fig = plt.figure(figsize=(10, 8))

ax1 = plt.subplot(2, 1, 1)
ax1.plot(TT_wind_sim_hourlyRamp.index, TT_wind_sim_hourlyRamp.values)
ax1.set_ylim([-0.2, 0.2])
ax1.set_title("Onshore wind generation hourly ramp (in GMT)")
ax1.legend(TT_wind_sim_hourlyRamp.columns)
ax1.set_xlabel("Time")
ax1.set_ylabel("Hourly Ramp")

ax2 = plt.subplot(2, 1, 2)
for col in TT_wind_sim_hourlyRamp.columns:
    ax2.hist(TT_wind_sim_hourlyRamp[col], bins=50, histtype='step', density=True)
ax2.set_xlim([-0.2, 0.2])
ax2.set_title("Hourly generation ramp PDF")
ax2.legend(TT_wind_sim_hourlyRamp.columns)
ax2.set_xlabel("Hourly Ramp")
ax2.set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
#run this to calculate ramp statistics

# Ramp statistics
T_ramp_stats_wind = pd.DataFrame({
    "Mean": TT_wind_sim_hourlyRamp.mean(),
    "SD": TT_wind_sim_hourlyRamp.std()
})

print("----- Ramp stats for wind ----")
print(T_ramp_stats_wind)

## Exercise 2: Wind and solar generation in multiple countries

We keep the data from Exercise 1 open and load also the simulated solar generation data for the three countries.

2.1: Run the cell to visualize the solar generation data as time series, on hourly resolution, daily resolution (daily mean) and monthly resolution (monthly mean). How does it look compared to wind generation? Why?

Answer: *Double click to edit*

In [ ]:

# ------------------------------------------------------------
# Exercise 2.1
# Visualization of solar generation
# ------------------------------------------------------------

# Load solar generation data
TT_solar_sim = pd.read_csv(folder_data + "Solar_generation_simulated.csv",
                           index_col=0, parse_dates=True)

# Compute aggregate solar generation from the 3 countries
TT_solar_sim["Aggregate"] = TT_solar_sim.iloc[:, 0:3].mean(axis=1)

# Daily and monthly resampled versions
TT_solar_sim_daily   = TT_solar_sim.resample("D").mean()
TT_solar_sim_monthly = TT_solar_sim.resample("ME").mean()

# Plotting 
fig = plt.figure(figsize=(10, 9))

ax1 = plt.subplot(3, 1, 1)
ax1.plot(TT_solar_sim.index, TT_solar_sim.values)
ax1.set_ylim(0, 1)
ax1.set_title("Solar generation (hourly, GMT)")
ax1.legend(TT_solar_sim.columns)
ax1.set_xlabel("Time")
ax1.set_ylabel("Capacity Factor")

ax2 = plt.subplot(3, 1, 2)
ax2.plot(TT_solar_sim_daily.index, TT_solar_sim_daily.values)
ax2.set_title("Solar generation (Daily)")
ax2.set_ylim(bottom=0)
ax2.legend(TT_solar_sim_daily.columns)
ax2.set_xlabel("Time")
ax2.set_ylabel("Capacity Factor")

ax3 = plt.subplot(3, 1, 3)
ax3.plot(TT_solar_sim_monthly.index, TT_solar_sim_monthly.values)
ax3.set_title("Solar generation (Monthly)")
ax3.set_ylim(bottom=0)
ax3.legend(TT_solar_sim_monthly.columns)
ax3.set_xlabel("Time")
ax3.set_ylabel("Capacity Factor")

plt.tight_layout()
plt.show()

**2.2:** Below we calculate correlations (Pearson correlation). Between all the wind generations. What can you say about the correlation? Why are some low and some high?

Answer: *Double click to edit*

**2.3:** Between all the solar generations. What can you say about the correlations? How do they appear compared to the wind generation correlations? Why do you think that is?

Answer: *Double click to edit*

**2.4:** Between all the wind and the solar generations. What can you say about the correlations between wind and solar generation?

Answer: *Double click to edit*

In [ ]:
# ------------------------------------------------------------
# Exercise 2.2, 2.3, 2.4:
# Combine wind + solar into one dataframe so that correlations
# (wind–wind, solar–solar, wind–solar) can be computed.
# ------------------------------------------------------------

# Combine wind + solar
TT_wind_temp  = TT_wind_sim.add_prefix("Wind_")
TT_solar_temp = TT_solar_sim.add_prefix("Solar_")

# Daily and monthly resampled versions
TT_wind_and_solar_sim = pd.concat(
    [TT_wind_temp.iloc[:, 0:3], TT_solar_temp.iloc[:, 0:3]],
    axis=1
)

TT_wind_and_solar_sim["Aggregate"] = TT_wind_and_solar_sim.mean(axis=1)

# Correlations
T_Cor = TT_wind_and_solar_sim.iloc[:, :6].corr()

# ------------------------------------------------------------
# Display results for correlations
# ------------------------------------------------------------

print("------- Correlations ---------")
print(T_Cor)

**2.5:** Here we calculate aggregate onshore wind and solar generation of the three countries, assuming each country has the same amount of installed capacity for both wind and solar (give the aggregate generation as standardized generation). 

Compare the RSD of this aggregate wind and solar generation to the aggregate wind generation from 1.5. How does it compare?

In [ ]:
# Stats for solar
T_stats_solar = pd.DataFrame({
    "Mean": TT_solar_sim.mean(),
    "SD": TT_solar_sim.std()
})
T_stats_solar["RSD"] = T_stats_solar["SD"] / T_stats_solar["Mean"]

# Combined wind+solar
TT_wind_and_solar = pd.DataFrame({
    "Aggregate_wind_and_solar":
        (TT_wind_sim["Aggregate"] + TT_solar_sim["Aggregate"]) / 2
}, index=TT_solar_sim.index)

TT_wind_and_solar_hourlyRamp = TT_wind_and_solar.diff()

T_stats_wind_and_solar = pd.DataFrame({
    "Mean": TT_wind_and_solar.mean(),
    "SD": TT_wind_and_solar.std()
})
T_stats_wind_and_solar["RSD"] = T_stats_wind_and_solar["SD"] / T_stats_wind_and_solar["Mean"]

print("----- Stats for wind and solar ----")
print(T_stats_wind_and_solar)

## Exercise 3: Compare and validate wind simulation models  
We keep the data from Exercise 1 open and load also the simulated wind generation data using a different model (model 2). We also load the measured wind generation data.

Run this cell to generate the plots for exercise 3.

In [ ]:
# ============================================================
# Exercise 3: Compare Models
# ============================================================

# Load model 2 and measured data
TT_wind_sim_model2 = pd.read_csv(folder_data + "Wind_generation_simulated_model2.csv",
                                 index_col=0, parse_dates=True)

TT_wind_meas = pd.read_csv(folder_data + "Wind_generation_measured.csv",
                           index_col=0, parse_dates=True)

country = "FI"

TT_model1 = TT_wind_sim[[country]].copy()
TT_model2 = TT_wind_sim_model2[[country]].copy()
TT_meas = TT_wind_meas[[country]].copy()

# Set simulated values to NaN where measurement is NaN
TT_model1[country] = TT_model1[country].where(~TT_meas[country].isna())
TT_model2[country] = TT_model2[country].where(~TT_meas[country].isna())

# Combined table
TT = pd.concat([
    TT_model1.rename(columns={country: "Model_1"}),
    TT_model2.rename(columns={country: "Model_2"}),
    TT_meas.rename(columns={country: "Meas"})
], axis=1)

# Daily/monthly
TT_daily = TT.resample("D").mean()
TT_monthly = TT.resample("ME").mean()

EMD_model1 = wasserstein_distance(
    TT["Model_1"].dropna(),
    TT["Meas"].dropna()
)
EMD_model2 = wasserstein_distance(
    TT["Model_2"].dropna(),
    TT["Meas"].dropna()
)

T_validation_stats = pd.DataFrame({
    "EMD": [EMD_model1, EMD_model2]
}, index=["Model_1", "Model_2"])

# Correlations
T_validation_stats["Cor_to_meas"] = [
    TT["Model_1"].corr(TT["Meas"]),
    TT["Model_2"].corr(TT["Meas"])
]

# Mean err
T_validation_stats["Mean_err"] = [
    TT["Model_1"].mean() - TT["Meas"].mean(),
    TT["Model_2"].mean() - TT["Meas"].mean()
]

# RMSE
rmse = lambda a, b: np.sqrt(np.nanmean((a - b)**2))

T_validation_stats["RMSE"] = [
    rmse(TT["Model_1"], TT["Meas"]),
    rmse(TT["Model_2"], TT["Meas"])
]

# Plot
fig = plt.figure(figsize=(12, 10))

ax1 = plt.subplot(3, 3, (1, 2))
ax1.plot(TT.index, TT.values)
ax1.set_title(f"Onshore wind generation (hourly, GMT): {country}")
ax1.set_ylim(0, 1)
ax1.legend(TT.columns)
ax1.set_xlabel("Time")
ax1.set_ylabel("Capacity Factor")

ax2 = plt.subplot(3, 3, (4, 5))
ax2.plot(TT_daily.index, TT_daily.values)
ax2.set_title("Onshore wind generation (Daily)")
ax2.legend(TT_daily.columns)
ax2.set_xlabel("Time")
ax2.set_ylabel("Capacity Factor")

ax3 = plt.subplot(3, 3, (7, 8))
ax3.plot(TT_monthly.index, TT_monthly.values)
ax3.set_title("Onshore wind generation (Monthly)")
ax3.legend(TT_monthly.columns)
ax3.set_xlabel("Time")
ax3.set_ylabel("Capacity Factor")

ax_hist = plt.subplot(3, 3, (3, 6))
for col in TT.columns:
    ax_hist.hist(TT[col].dropna(), bins=40, density=True, histtype="step")
ax_hist.set_xlim(0, 1)
ax_hist.set_title("Generation PDF (hourly)")
ax_hist.legend(TT.columns)
ax_hist.set_xlabel("Capacity Factor")
ax_hist.set_ylabel("Density")

plt.tight_layout()
plt.show()

print("----- Validation stats ----")
print(T_validation_stats)

**3.1:** Consider the figures of the two simulation models and the measured for Finland (Meas), on hourly resolution, daily resolution (daily mean) and monthly resolution (monthly mean). How well would you say the two models represent the measured data?  

Answer: *Double click to edit*

**3.2:** Compare the two models regarding validation to the measured data in terms of RMSE (hourly data). Which model looks better?  

Answer: *Double click to edit*

**3.3:** Compare the two models regarding validation to the measured data in terms of distributional properties (look at the PDFs, calculate EMD metrics, look at very low/high generation value likelihoods, etc.). Which model looks better?  

Answer: *Double click to edit*

**3.4:** Compare the two models regarding correlation to the measured data. Which model looks better? 

Answer: *Double click to edit*

**3.5:** Compare the two models in getting the annual production similar to the measured data. Which model looks better?

Answer: *Double click to edit*